# Testing corrected spectral metrics

Fake peaks in the psd spectrum of a node are created when nearby nodes overlap in the frequency domain. Corrected metrics remove peaks by checking where the psd of the reconstructed node is above the original signal and pulling it down in line with the original before normalising and calculating metrics.

In [ ]:
import sys
sys.path.append("..")
if sys.version_info[0] + 0.1 * sys.version_info[1] < 3.6:
    raise Exception("Must be using Python 3.6+ version")

import scipy.ndimage as ndi
import plotly.graph_objects as go
import numpy as np

In [ ]:
from wavelet_transform.wpt.transform import dwpt
import wavelet_transform.utils.bubble_data_manager as bdm
from wavelet_transform.utils.preprocessing import preprocess, add_noise, len_full_cycles
from wavelet_transform.utils.extensions import to_dB, psd
import wavelet_transform.wpt.inverse_transform as inv
from scipy.stats import kurtosis

### Get bubble dynamics data

In [ ]:
# Set the sonication pulse parameters
pac = 1e6  # the amplitude of the sine wave in Pa
frequency = 1.1 * 1e6  # insonation frequency (Hz)
period = 1 / frequency  # the period of the acoustic wave at the fundamental frequency 
n_cycles = 100  # no. of cycles of the sine wave
parameters = {
    "time_delay": 0 * period,  # time delay, t1
    "pulse_width": 1000 * n_cycles * period,  # pulse width, W = 1 / Q
    "phase_initial": 0 * period,  # tau
    "phase_shift": 0,  # np.pi / 4,  # phi
    "no_harmonics": 1,   # k
    "Rdot_init": 0,  # initial bubble's wall velocity in m/s
}

# Set sampling rate and noise std
sampling_rate = 200e6  # sampling rate to generate the time vector
signal2noise = 20

r_measure = 1e-4  # distance from the centre of bubble which radiated pressure (acoustic emission from bubble) is calculated over time

Rinit = 3.4e-6  # initial bubble radius in m

In [ ]:
data, t_plot = bdm.get_data(
    Rinit, frequency, pac, sampling_rate, n_cycles=n_cycles, r_measure=r_measure,
    return_time=True, **parameters
)

# Add noise
np.random.seed(64)
data = add_noise(data, signal2noise)
np.random.seed(None)

### DWPT params

In [ ]:
packet_bandwidth = frequency / 4  # bandwidth of the excitation pulse
required_levels = int(np.ceil(np.log2(sampling_rate / 2 / packet_bandwidth)))
print(f"Required levels: {required_levels}")

In [ ]:
# Parameters for the wavelet transform
wavelet = "sym20"
levels = required_levels  # set to None to use max-level transform
mode = "zero"
best_basis = "energy"
min_level = 0
freq_threshold = min(1, 100e6 / (sampling_rate / 2)) # the nodes with norm_freq_range above this threshold will not be kept
fft_lim = 10  # this is the upper limit for plotting the spectrum (fft/psd of emissions)

threshold = 3.2

### Preprocess data

In [ ]:
# Normalise. Trim experimental data
data_preprocessed = preprocess(data, window=False)
data_preprocessed = data_preprocessed[:int(len_full_cycles(sampling_rate, frequency, len(data_preprocessed)))]
t_plot = t_plot[:len(data_preprocessed)]

print("samples before trimming:", len(data))
print("samples after trimming:", len(data_preprocessed))

In [ ]:
keep_ratio = 10e3 / len(data_preprocessed)  # coefficients kept to calculate statistical metrics directly on wavelet coefficients
inv_keep_ratio = keep_ratio  # the ratio used for inverse transform metrics

assert keep_ratio >= 10e3 / len(data_preprocessed), (
    f"""
    current keep ratio: {keep_ratio}, 
    min keep ratio for 10k samples: {10e3 / len(data_preprocessed)}
    """
)

### DWPT

In [ ]:
tr_obj = dwpt(
    data_preprocessed,
    levels,
    wavelet,
    mode,
    best_basis=best_basis,
    min_level=min_level,
    freq_threshold=freq_threshold
)
tr_obj.keep_ratio = keep_ratio

inv_tr_obj = inv.InverseTransform(tr_obj)

### Plot metrics

In [ ]:
metrics_to_test = [
    "spectral kurtosis",  # x axis
    "corrected spectral kurtosis",  # y axis
]

wpt_df_test = inv_tr_obj.info(
    freq_threshold=freq_threshold, keep_ratio=inv_keep_ratio,
    metrics=metrics_to_test, include_node_info=True
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=wpt_df_test[metrics_to_test[0]],
        y=wpt_df_test[metrics_to_test[1]],
        mode="markers",
    )
)
fig.add_trace(
    go.Scatter(
        x=np.arange(0, max(wpt_df_test[metrics_to_test[0]]), 0.01),
        y=np.arange(0, max(wpt_df_test[metrics_to_test[0]]), 0.01),
        mode="lines",
        line=dict(color="red", dash="dash"),
        name="y=x",
    )
)
fig.update_layout(
    title=f"{metrics_to_test[1]} vs {metrics_to_test[0]}",
    xaxis_title=metrics_to_test[0],
    yaxis_title=metrics_to_test[1],
)
fig.show()

In [ ]:
wpt_df_test

In [ ]:
node = tr_obj.get_node(9, 7)
rec = inv.inv_node(node)
psd_rec, freq = psd(rec, sampling_rate, frequency)
psd_rec_corrected = np.minimum(tr_obj.ref_psd, psd_rec)
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=freq,
        y=to_dB(tr_obj.ref_psd),
        mode="lines",
        name="Original PSD",
    )
)
fig.add_trace(
    go.Scatter(
        x=freq,
        y=to_dB(psd_rec),
        mode="lines",
        name="Reconstructed PSD",
        line=dict(dash="dot"),
    )
)
fig.add_trace(
    go.Scatter(
        x=freq,
        y=to_dB(psd_rec_corrected),
        mode="lines",
        name="Corrected Reconstructed PSD",
        line=dict(dash="dash"),
    )
)
fig.update_layout(
    title=f"Node {node.level, node.node_number} PSD Reconstruction",
    xaxis_title="F / F0",
    yaxis_title="PSD (dB)",
    xaxis=dict(range=[0, fft_lim]),
)
fig.show()

print("reconstructed kurtosis", kurtosis(psd_rec))
print("corrected reconstructed kurtosis", kurtosis(psd_rec_corrected))

In [ ]:
signal = np.zeros(1024)  # Placeholder to avoid errors
signal_transient = signal.copy()
signal_transient[len(signal)//2] = 1

signal_double_transient = signal_transient.copy()
signal_double_transient[len(signal)//4] = 1

signal_triple_transient = signal_double_transient.copy()
signal_triple_transient[3*len(signal)//4] = 1

print("kurtosis signal", kurtosis(signal))
print("kurtosis single transient", kurtosis(signal_transient))
print("kurtosis double transient", kurtosis(signal_double_transient))
print("kurtosis triple transient", kurtosis(signal_triple_transient))